In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import itertools
from dgl.nn import SAGEConv, GraphConv, GATConv
import dgl.function as fn
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
import utils

np.set_printoptions(suppress=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')


In [ ]:
DEBUG = True
datasets, seeds, percentages = utils.get_config(DEBUG)


In [ ]:
def process_edges(edges_chunk, edges_in_graph):
    pos_u = []
    pos_v = []
    neg_u = []
    neg_v = []
    for u, v in edges_chunk:
        if (u, v) in edges_in_graph:
            pos_u.append(u)
            pos_v.append(v)
        else:
            neg_u.append(u)
            neg_v.append(v)
    return pos_u, pos_v, neg_u, neg_v

class DotPredictor(nn.Module):
    def forward(self, g, h):
        with g.local_scope():
            g.ndata['h'] = h
            g.apply_edges(fn.u_dot_v('h', 'h', 'score'))
            return g.edata['score'][:, 0]


In [ ]:
class GAT(nn.Module):
    def __init__(self, in_feats, h_feats, num_heads=4):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_feats, h_feats, num_heads=num_heads, allow_zero_in_degree=True)
        self.conv2 = GATConv(h_feats * num_heads, h_feats, num_heads=1, allow_zero_in_degree=True)

    def forward(self, g, in_feat):
        h = self.conv1(g, in_feat)
        h = h.flatten(1)
        h = F.elu(h)
        h = self.conv2(g, h)
        h = h.squeeze(1)
        return h


In [ ]:
# Main processing loops
for seed in seeds:
    print(f'Processing seed {seed}')
    for ds in datasets:
        print(f'Analyzing {ds} dataset with seed {seed}')

        path = f'./data/w_removal_{ds}'
        if not os.path.exists(path):
            print(f"Data file not found at {path}. Skipping.")
            continue

        df = pd.read_csv(path, sep=' ', header=None,
                         names=["n1","n2","f1","f2","f3","f4","f5","f6","f7","l"])
        df.dropna(subset=['n1','n2'], inplace=True)

        # Node indexing
        nodes = np.unique(df[['n1','n2']].values)
        idx   = {n:i for i,n in enumerate(nodes)}

        # Build original graph of positives
        pos = df[df.l==1]
        src = torch.tensor(pos.n1.map(idx).values, dtype=torch.int64)
        dst = torch.tensor(pos.n2.map(idx).values, dtype=torch.int64)
        g   = dgl.graph((torch.cat([src,dst]), torch.cat([dst,src])),
                        num_nodes=len(nodes)).to(device)

        # Enumerate all undirected node-pairs
        adj = pd.DataFrame(0, index=range(len(nodes)), columns=range(len(nodes)))
        for _,r in df.iterrows():
            i,j = idx[r.n1], idx[r.n2]
            adj.iat[i,j] = adj.iat[j,i] = 1
            
        ui,vi = np.triu_indices_from(adj.values, k=1)
        all_edges = [(i,j) for i,j in zip(ui,vi)] + [(j,i) for i,j in zip(ui,vi)]

        np.random.seed(seed)
        np.random.shuffle(all_edges)

        existing = {(u.item(),v.item()) for u,v in zip(*g.edges())}

        for p in percentages:
            pct_int = int(p * 100)
            print(f'  ▶ test/train split: {pct_int}% test')

            n_test = int(len(all_edges) * p)
            test_edges = set(all_edges[:n_test])
            train_edges = all_edges[n_test:]

            test_pos_u, test_pos_v, test_neg_u, test_neg_v = process_edges(test_edges, existing)
            test_pos_g = dgl.graph((torch.tensor(test_pos_u), torch.tensor(test_pos_v)),
                                   num_nodes=g.num_nodes()).to(device)
            test_neg_g = dgl.graph((torch.tensor(test_neg_u), torch.tensor(test_neg_v)),
                                   num_nodes=g.num_nodes()).to(device)

            eids = [ eid for eid,(u,v) in enumerate(zip(*g.edges()))
                     if (u.item(),v.item()) in test_edges ]
            train_g = dgl.remove_edges(g, eids).to(device)

            train_pos_u, train_pos_v, train_neg_u, train_neg_v = process_edges(train_edges, existing)
            train_pos_g = dgl.graph((torch.tensor(train_pos_u), torch.tensor(train_pos_v)),
                                    num_nodes=train_g.num_nodes()).to(device)
            train_neg_g = dgl.graph((torch.tensor(train_neg_u), torch.tensor(train_neg_v)),
                                    num_nodes=train_g.num_nodes()).to(device)

            inputs = train_g.adj().to_dense().to(device)

            # Build model
            model = GAT(train_g.num_nodes(), 16).to(device)
            pred  = DotPredictor().to(device)
            optimizer = torch.optim.Adam(itertools.chain(model.parameters(), pred.parameters()), lr=0.0005)

            num_epochs = 1500
            if DEBUG:
                num_epochs = 2
                
            model.train()
            for epoch in range(num_epochs):
                h = model(train_g, inputs)
                pos_score = pred(train_pos_g, h)
                neg_score = pred(train_neg_g, h)
                loss = utils.compute_loss_gnn(pos_score, neg_score, device)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                if epoch % 100 == 0 or (DEBUG and epoch == 1):
                    print(f'    Epoch {epoch:4d} Loss: {loss.item():.4f}')

            model.eval()
            with torch.no_grad():
                h = model(train_g, inputs)
                pos_score = pred(test_pos_g, h)
                neg_score = pred(test_neg_g, h)
                
                auc = utils.compute_auc_gnn(pos_score, neg_score)
                acc = utils.compute_acc_gnn(pos_score, neg_score)
                
                scores = torch.cat([pos_score, neg_score]).cpu().detach().numpy()
                preds = (torch.sigmoid(torch.tensor(scores)) >= 0.5).numpy()
                labels = torch.cat([torch.ones(pos_score.shape[0]), torch.zeros(neg_score.shape[0])]).cpu().detach().numpy()
                
                cm = confusion_matrix(labels, preds)
                if cm.shape == (2, 2):
                    ber = utils.balanced_error_rate(cm)
                else:
                    ber = 0.5
                print(f'    → AUC: {auc:.4f}  ACC: {acc:.4f}  BER: {ber:.4f}')

            base_dir = f'./results/GAT/{ds}'
            dirs = {
                'cm_png':  os.path.join(base_dir, 'cm_png',  str(seed), str(pct_int)),
                'cm_npy':  os.path.join(base_dir, 'cm_npy',  str(seed), str(pct_int)),
                'models':  os.path.join(base_dir, 'models',  str(seed), str(pct_int)),
                'csv':     os.path.join(base_dir, 'csv',     str(seed), str(pct_int)),
            }
            for path_dir in dirs.values():
                os.makedirs(path_dir, exist_ok=True)

            if cm.shape == (2,2):
                plt.figure(figsize=(6,6))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
                plt.title(f'CM {ds} seed{seed} {pct_int}%')
                plt.savefig(os.path.join(dirs['cm_png'], f'cm_{ds}_seed{seed}_p{pct_int}.png'))
                plt.close()
                np.save(os.path.join(dirs['cm_npy'], f'cm_{ds}_seed{seed}_p{pct_int}.npy'), cm)

            torch.save(model.state_dict(), os.path.join(dirs['models'], f'model_{ds}_seed{seed}_p{pct_int}.pth'))

            res = [{'Dataset': ds, 'Seed': seed, 'Percentage': p, 'AUC': auc, 'ACC': acc, 'BER': ber}]
            pd.DataFrame(res).to_csv(os.path.join(dirs['csv'], f'results_{ds}_seed{seed}_p{pct_int}.csv'), index=False)
